In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l4.4 Transformer block

Now stack the pieces into the real PocketLM block: pre-norm + causal attention +
residual, then pre-norm + MLP + residual. The whole model is just N of these in
a row.

In [ ]:
from pocketlm.model import Block, PocketLMConfig

torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=11, block_size=8, n_layer=1, n_head=2, n_embd=16)
block = Block(cfg).eval()
x = torch.randn(1, 5, 16)
out = block(x)
print("block in", tuple(x.shape), "-> out", tuple(out.shape),
      "(shape preserved, so blocks stack)")

Because the block preserves shape, you can stack any depth. Depth is where
abstraction comes from: early blocks see characters, later ones see structure.

In [ ]:
assert tuple(out.shape) == (1, 5, 16)